In [1]:
"""
Data Preprocessing Script
Process raw dataset: clean price, rating, text, handle missing values
"""

import pandas as pd
import re

# ----------------------------- Helper Functions -----------------------------
def parse_rating(rating_value):
    """
    Convert textual or numeric rating to float.
    Handles:
    - 'One', 'Two', ... → 1,2,...
    - Numeric strings → float
    - Already numeric → float
    - None/NaN → None
    """
    if pd.isna(rating_value):
        return None

    # If already numeric, return float
    if isinstance(rating_value, (int, float)):
        return float(rating_value)

    rating_dict = {'one': 1, 'two': 2, 'three': 3, 'four': 4, 'five': 5}
    
    rating_str = str(rating_value).strip().lower()

    # Try textual conversion
    if rating_str in rating_dict:
        return float(rating_dict[rating_str])

    # Try numeric conversion
    try:
        return float(rating_str)
    except ValueError:
        return None

def clean_price(price_text):
    """Convert formatted price to float"""
    if price_text is None or pd.isna(price_text):
        return None
    price_clean = re.sub(r"[^0-9.]", "", str(price_text))
    try:
        return float(price_clean)
    except ValueError:
        return None

def clean_text(text):
    """Remove special characters & normalize lowercase"""
    if not text or pd.isna(text):
        return None
    return re.sub(r"[^a-z0-9\s]", "", str(text).lower()).strip()

In [5]:
# ----------------------------- Load Raw Data -----------------------------
df = pd.read_csv("../data_collection/raw_book_data.csv")
print("Initial shape:", df.shape)

Initial shape: (1050, 6)


In [6]:
# ----------------------------- Cleaning -----------------------------

# Drop duplicates
df = df.drop_duplicates(subset=['title', 'source'])

# Apply cleaning functions
df['price'] = df['price'].apply(clean_price)
df['rating'] = df['rating'].apply(parse_rating)

df['title'] = df['title'].apply(clean_text)
df['category'] = df['category'].apply(clean_text)

# Normalize availability labels
df['availability'] = (
    df['availability']
    .astype(str)
    .str.lower()
    .str.strip()
    .replace({
        "instock": "in stock",
        "in stock": "in stock",
        "outofstock": "out of stock",
        "out of stock": "out of stock"
    })
)

# Handle missing values safely
df['price'] = pd.to_numeric(df['price'], errors='coerce').fillna(0).astype(float)
df['rating'] = pd.to_numeric(df['rating'], errors='coerce').fillna(0).astype(float)
df['category'] = df['category'].fillna("unknown").astype(str)
df['availability'] = df['availability'].fillna("unknown").astype(str)

# Remove rows with missing/blank titles
df = df[df['title'].notna() & (df['title'].str.strip() != "")]

In [7]:
# ----------------------------- Save Cleaned Data -----------------------------
df.to_csv("cleaned_book_data.csv", index=False, encoding="utf-8-sig")
print("Data Cleaned and Preprocessed Successfully.")
print(df.head())

Data Cleaned and Preprocessed Successfully.
           source                                 title  price  rating  \
0  books.toscrape                  a light in the attic  51.77     3.0   
1  books.toscrape                    tipping the velvet  53.74     1.0   
2  books.toscrape                            soumission  50.10     1.0   
3  books.toscrape                         sharp objects  47.82     4.0   
4  books.toscrape  sapiens a brief history of humankind  54.23     5.0   

             category availability  
0              poetry     in stock  
1  historical fiction     in stock  
2             fiction     in stock  
3             mystery     in stock  
4             history     in stock  
